In [ ]:
# ==========================================
# CELL 1: IMPORTS, REPRODUCIBILITY & SETUP
# ==========================================
import os
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings
warnings.filterwarnings('ignore')

SEED = 999
np.random.seed(SEED)
torch.manual_seed(SEED)

# Output Directories (Strict Naming)
DIRS = ['results/tables', 'results/figures', 'results/models', 'results/logs']
for d in DIRS:
    os.makedirs(d, exist_ok=True)

# Hardware Optimization
GPU_AVAILABLE = torch.cuda.is_available()
MAX_WORKERS = min(os.cpu_count() - 1, 4) if hasattr(os, 'cpu_count') and os.cpu_count() else 4
DEVICE = 'cuda' if GPU_AVAILABLE else 'cpu'

print(f"✓ Seed fixed to {SEED}")
print(f"✓ GPU Acceleration: {GPU_AVAILABLE}")
print(f"✓ Parallel Workers: {MAX_WORKERS}")
print(f"✓ Clean Directories Created (including logs)")

In [ ]:
# ==========================================
# CELL 2: CORE TRAINING ENGINE
# ==========================================

def optimized_trainer(X, y, task_type, is_bert):
    """Trains an XGBoost model with automatic class balancing for binary tasks"""
    
    X_final, y_final = X, y

    if task_type == 'binary':
        human_idx = y[y == 0].index
        ai_idx = y[y == 1].index
        
        n_human = len(human_idx)
        n_ai = len(ai_idx)
        
        # since the dataset is imbalanced with more AI samples
        if n_human < n_ai:
            random_ai_idx = np.random.RandomState(SEED).choice(ai_idx, size=n_human, replace=False)
            balanced_idx = np.concatenate([human_idx, random_ai_idx])
            
            X_final = X.loc[balanced_idx]
            y_final = y.loc[balanced_idx]
            print(f"  [Balance] Binary Task: Balanced to {n_human} Human vs {n_human} AI samples.")
    
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_final, y_final, test_size=0.20, random_state=SEED, stratify=y_final
    )
    
    #  Normalization
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype('float32')
    X_test_scaled = scaler.transform(X_test).astype('float32')
    
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
    X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)
    
    # Model Configuration
    objective = 'binary:logistic' if task_type == 'binary' else 'multi:softprob'
    
    params = {
        'objective': objective,
        'tree_method': 'hist',
        'device': DEVICE,
        'n_estimators': 2000,
        'early_stopping_rounds': 100,
        'learning_rate': 0.01,
        'max_depth': 10,
        'random_state': SEED,
        'n_jobs': MAX_WORKERS
    }
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_train_scaled, y_train, eval_set=[(X_test_scaled, y_test)], verbose=False)
    
    # Predictions
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average=None, zero_division=0)
    
    return {
        'model': model, 'scaler': scaler, 'acc': acc, 
        'prec': prec, 'rec': rec, 'f1': f1,
        'y_test': y_test, 'y_pred': y_pred, 'features': X.columns,
        'params': params 
    }

In [ ]:
# ==========================================
# CELL 3: TASK 1.1 - BASELINE & ARTIFACTS
# ==========================================
print("Loading Main Dataset...")
df = pd.read_csv('data/processed/processed_articles.csv')
df = df.drop(columns=['Text'], errors='ignore')

# Separate Features
bert_cols = [c for c in df.columns if c.startswith('bert_')]
X_all = df.drop(columns=['is_AI', 'Writer'])
X_no_bert = X_all.drop(columns=bert_cols)

# Encoders
le_multi = LabelEncoder()
y_bin = df['is_AI']
y_multi = le_multi.fit_transform(df['Writer'])
multi_classes = list(le_multi.classes_)

# Parallel Training
# bin = binary, multi = multiclass, NoBERT = features only, BERT = features + BERT
tasks = {
    'Bin_NoBERT': (X_no_bert, y_bin, 'binary', False),
    'Bin_BERT': (X_all, y_bin, 'binary', True),
    'Multi_NoBERT': (X_no_bert, y_multi, 'multiclass', False),
    'Multi_BERT': (X_all, y_multi, 'multiclass', True)
}

results = {}
print("Starting Parallel Training for 4 Configurations...")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(optimized_trainer, *args): name for name, args in tasks.items()}
    for future in as_completed(futures):
        name = futures[future]
        results[name] = future.result()
        print(f"✓ {name} Completed - Acc: {results[name]['acc']:.4f}")
        
        # Save Models & Scalers
        joblib.dump(results[name]['model'], f"results/models/{name}_model.joblib")
        joblib.dump(results[name]['scaler'], f"results/models/{name}_scaler.joblib")
        
        # Save Metadata
        metadata = {
            "seed": SEED,
            "data_split": "80/20",
            "task_type": "binary" if "Bin" in name else "multiclass",
            "feature_list": list(results[name]['features'])
        }
        with open(f"results/models/{name}_metadata.json", 'w') as f:
            json.dump(metadata, f, indent=4)

# Save Hyperparameters Log
sample_params = results['Bin_BERT']['params']
with open('results/logs/xgboost_hyperparameters.json', 'w') as f:
    json.dump(sample_params, f, indent=4)
print("✓ Models, Metadata, and Hyperparameters Log Saved.")

# --- GENERATE TABLES ---
# T1: Binary Results (Per-Class)
t1_df = pd.DataFrame([
    {'Configuration': 'Features only', 
     'Accuracy': results['Bin_NoBERT']['acc'],
     'Human_Precision': results['Bin_NoBERT']['prec'][0], 
     'Human_Recall': results['Bin_NoBERT']['rec'][0], 
     'Human_F1': results['Bin_NoBERT']['f1'][0],
     'AI_Precision': results['Bin_NoBERT']['prec'][1], 
     'AI_Recall': results['Bin_NoBERT']['rec'][1], 
     'AI_F1': results['Bin_NoBERT']['f1'][1]},
     
    {'Configuration': 'Features + BERT', 
     'Accuracy': results['Bin_BERT']['acc'],
     'Human_Precision': results['Bin_BERT']['prec'][0], 
     'Human_Recall': results['Bin_BERT']['rec'][0], 
     'Human_F1': results['Bin_BERT']['f1'][0],
     'AI_Precision': results['Bin_BERT']['prec'][1], 
     'AI_Recall': results['Bin_BERT']['rec'][1], 
     'AI_F1': results['Bin_BERT']['f1'][1]}
])
t1_df.round(4).to_csv('results/tables/T1.csv', index=False)

# T2: Multi-class Results (Per-Class)
t2_data = []
for config_name, res_key in [('Features only', 'Multi_NoBERT'), ('Features + BERT', 'Multi_BERT')]:
    row = {'Configuration': config_name, 'Accuracy': results[res_key]['acc']}
    for i, cls in enumerate(multi_classes):
        row[f'{cls}_Precision'] = results[res_key]['prec'][i]
        row[f'{cls}_Recall'] = results[res_key]['rec'][i]
        row[f'{cls}_F1'] = results[res_key]['f1'][i]
    t2_data.append(row)

t2_df = pd.DataFrame(t2_data)
t2_df.round(4).to_csv('results/tables/T2.csv', index=False)

# T3: Per-class F1 Multi-class
t3_df = pd.DataFrame({'Config': ['Features only', 'Features + BERT']})
for i, cls in enumerate(multi_classes):
    t3_df[cls] = [results['Multi_NoBERT']['f1'][i], results['Multi_BERT']['f1'][i]]
t3_df.round(4).to_csv('results/tables/T3.csv', index=False)

# --- GENERATE FIGURES ---
def plot_cm(y_true, y_pred, labels, filename, title):
    plt.figure(figsize=(10, 8), dpi=300)
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.savefig(f'results/figures/{filename}', bbox_inches='tight')
    plt.close()

def plot_imp(model, features, filename, title):
    imp = pd.DataFrame({'Feature': features, 'Importance': model.feature_importances_}).sort_values('Importance', ascending=False).head(20)
    plt.figure(figsize=(12, 8), dpi=300)
    sns.barplot(x='Importance', y='Feature', data=imp, palette='viridis')
    plt.title(title)
    plt.savefig(f'results/figures/{filename}', bbox_inches='tight')
    plt.close()

plot_cm(results['Bin_BERT']['y_test'], results['Bin_BERT']['y_pred'], ['Human', 'AI'], 'F1.png', 'F1: Binary CM (Features+BERT)')
plot_cm(results['Multi_BERT']['y_test'], results['Multi_BERT']['y_pred'], multi_classes, 'F2.png', 'F2: Multi-class CM (Features+BERT)')

plot_imp(results['Bin_NoBERT']['model'], X_no_bert.columns, 'F3.png', 'F3: Binary Importance (Features Only)')
plot_imp(results['Bin_BERT']['model'], X_all.columns, 'F4.png', 'F4: Binary Importance (Features+BERT)')
plot_imp(results['Multi_NoBERT']['model'], X_no_bert.columns, 'F5.png', 'F5: Multi-class Importance (Features Only)')
plot_imp(results['Multi_BERT']['model'], X_all.columns, 'F6.png', 'F6: Multi-class Importance (Features+BERT)')

print("✓ Task 1.1 Complete. Tables T1-T3 and Figures F1-F6 saved.")

In [ ]:
# ==========================================
# CELL 4: TASK 1.2 - MISSPELLING SWEEP
# ==========================================
print("Running Misspelling Sweep...")
rates = [5, 10, 15, 20]
t4_data = []

# Without perturbation (baseline)
t4_data.append({
    'Perturbation %': '0% (baseline)',
    'Bin Acc (Features-only)': results['Bin_NoBERT']['acc'],
    'Bin F1 (Features-only)': results['Bin_NoBERT']['f1'].mean(),
    'Bin Acc (Features+BERT)': results['Bin_BERT']['acc'],
    'Bin F1 (Features+BERT)': results['Bin_BERT']['f1'].mean(),
    'Multi Acc (Features-only)': results['Multi_NoBERT']['acc'],
    'Multi F1 (Features-only)': results['Multi_NoBERT']['f1'].mean(),
    'Multi Acc (Features+BERT)': results['Multi_BERT']['acc'],
    'Multi F1 (Features+BERT)': results['Multi_BERT']['f1'].mean()
})

multi_acc_nobert, multi_acc_bert = [results['Multi_NoBERT']['acc']], [results['Multi_BERT']['acc']]
bin_acc_nobert, bin_acc_bert = [results['Bin_NoBERT']['acc']], [results['Bin_BERT']['acc']]

for rate in rates:
    df_miss = pd.read_csv(f'data/processed/processed_misspelled_{rate}.csv')
    df_miss = df_miss.drop(columns=['Text'], errors='ignore')
    
    X_miss_all = df_miss.drop(columns=['is_AI', 'Writer'])
    X_miss_nobert = X_miss_all.drop(columns=bert_cols)
    y_bin_miss = df_miss['is_AI']
    y_multi_miss = le_multi.transform(df_miss['Writer'])
    
    # Transform
    X_miss_nobert_sc = pd.DataFrame(results['Bin_NoBERT']['scaler'].transform(X_miss_nobert), columns=X_miss_nobert.columns).astype('float32')
    X_miss_all_sc = pd.DataFrame(results['Bin_BERT']['scaler'].transform(X_miss_all), columns=X_miss_all.columns).astype('float32')
    
    # Predict
    pred_bin_nobert = results['Bin_NoBERT']['model'].predict(X_miss_nobert_sc)
    pred_bin_bert = results['Bin_BERT']['model'].predict(X_miss_all_sc)
    pred_multi_nobert = results['Multi_NoBERT']['model'].predict(X_miss_nobert_sc)
    pred_multi_bert = results['Multi_BERT']['model'].predict(X_miss_all_sc)
    
    # Store for Degradation Curve
    multi_acc_nobert.append(accuracy_score(y_multi_miss, pred_multi_nobert))
    multi_acc_bert.append(accuracy_score(y_multi_miss, pred_multi_bert))
    bin_acc_nobert.append(accuracy_score(y_bin_miss, pred_bin_nobert))
    bin_acc_bert.append(accuracy_score(y_bin_miss, pred_bin_bert))
    
    # calculate metrics for features-only model
    b_prec_no, b_rec_no, b_f1_no, _ = precision_recall_fscore_support(y_bin_miss, pred_bin_nobert, average='macro', zero_division=0)
    m_prec_no, m_rec_no, m_f1_no, _ = precision_recall_fscore_support(y_multi_miss, pred_multi_nobert, average='macro', zero_division=0)

    # calculate metrics for BERT model
    b_prec, b_rec, b_f1, _ = precision_recall_fscore_support(y_bin_miss, pred_bin_bert, average='macro', zero_division=0)
    m_prec, m_rec, m_f1, _ = precision_recall_fscore_support(y_multi_miss, pred_multi_bert, average='macro', zero_division=0)
    
    # add the complete results to T4 table to prove the impact of the spelling attack
    t4_data.append({
        'Perturbation %': f'{rate}%',
        'Bin Acc (Features-only)': accuracy_score(y_bin_miss, pred_bin_nobert),
        'Bin F1 (Features-only)': b_f1_no,
        'Bin Acc (Features+BERT)': accuracy_score(y_bin_miss, pred_bin_bert),
        'Bin F1 (Features+BERT)': b_f1,
        'Multi Acc (Features-only)': accuracy_score(y_multi_miss, pred_multi_nobert),
        'Multi F1 (Features-only)': m_f1_no,
        'Multi Acc (Features+BERT)': accuracy_score(y_multi_miss, pred_multi_bert),
        'Multi F1 (Features+BERT)': m_f1
    })

# T4 Table
pd.DataFrame(t4_data).round(4).to_csv('results/tables/T4.csv', index=False)

# F7 & F8: Degradation Curves
def plot_degradation(y1, y2, title, filename):
    plt.figure(figsize=(8, 6), dpi=300)
    x_labels = ['0%', '5%', '10%', '15%', '20%']
    plt.plot(x_labels, y1, marker='o', label='Features Only')
    plt.plot(x_labels, y2, marker='s', label='Features + BERT')
    plt.title(title)
    plt.ylabel('Accuracy')
    plt.xlabel('Perturbation Rate')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.savefig(f'results/figures/{filename}', bbox_inches='tight')
    plt.close()

plot_degradation(bin_acc_nobert, bin_acc_bert, 'F7: Binary Misspelling Degradation', 'F7.png')
plot_degradation(multi_acc_nobert, multi_acc_bert, 'F8: Multi-class Misspelling Degradation', 'F8.png')

print("✓ Task 1.2 Complete. Table T4 and Figures F7, F8 saved.")

In [ ]:
# ==========================================
# CELL 5: TASK 1.3 - LENGTH SENSITIVITY
# ==========================================
print("Running Length Sensitivity Analysis...")

# 1. Load the original text to compute Word_Count (since Text was dropped in df earlier)
df_full_text = pd.read_csv('data/processed/processed_articles.csv', usecols=['Text', 'is_AI', 'Writer'])

# 2. Extract exactly the same Test splits used in the headline run using the saved indices in `results`
bin_test_indices = results['Bin_BERT']['y_test'].index
multi_test_indices = results['Multi_BERT']['y_test'].index

df_test_bin = df_full_text.loc[bin_test_indices].copy()
df_test_multi = df_full_text.loc[multi_test_indices].copy()

# 3. Compute Word Counts and Bins separately
df_test_bin['Word_Count'] = df_test_bin['Text'].astype(str).apply(lambda x: len(x.split()))
df_test_bin['Length_Bin'] = pd.qcut(df_test_bin['Word_Count'], q=3, labels=['Short', 'Medium', 'Long'])

df_test_multi['Word_Count'] = df_test_multi['Text'].astype(str).apply(lambda x: len(x.split()))
df_test_multi['Length_Bin'] = pd.qcut(df_test_multi['Word_Count'], q=3, labels=['Short', 'Medium', 'Long'])

t5_data = []

# 4. Evaluate per Length Bin
for bin_label in ['Short', 'Medium', 'Long']:
    
    # --- Binary ---
    bin_mask = (df_test_bin['Length_Bin'] == bin_label)
    bin_idx = df_test_bin[bin_mask].index
    
    y_bin_true = results['Bin_BERT']['y_test'].loc[bin_idx]
    
    # Get the features for these specific test samples using memory variables
    X_bin_nobert = X_no_bert.loc[bin_idx]
    X_bin_all = X_all.loc[bin_idx]
    
    # Scale
    X_bin_nobert_sc = pd.DataFrame(results['Bin_NoBERT']['scaler'].transform(X_bin_nobert), columns=X_bin_nobert.columns).astype('float32')
    X_bin_all_sc = pd.DataFrame(results['Bin_BERT']['scaler'].transform(X_bin_all), columns=X_bin_all.columns).astype('float32')
    
    # Predict
    p_bin_nobert = results['Bin_NoBERT']['model'].predict(X_bin_nobert_sc)
    p_bin_bert = results['Bin_BERT']['model'].predict(X_bin_all_sc)
    
    # --- Multi-class ---
    multi_mask = (df_test_multi['Length_Bin'] == bin_label)
    multi_idx = df_test_multi[multi_mask].index
    
    y_multi_true = results['Multi_BERT']['y_test'].loc[multi_idx]
    
    # Get the features for these specific test samples using memory variables
    X_multi_nobert = X_no_bert.loc[multi_idx]
    X_multi_all = X_all.loc[multi_idx]
    
    # Scale
    X_multi_nobert_sc = pd.DataFrame(results['Multi_NoBERT']['scaler'].transform(X_multi_nobert), columns=X_multi_nobert.columns).astype('float32')
    X_multi_all_sc = pd.DataFrame(results['Multi_BERT']['scaler'].transform(X_multi_all), columns=X_multi_all.columns).astype('float32')
    
    # Predict
    p_multi_nobert = results['Multi_NoBERT']['model'].predict(X_multi_nobert_sc)
    p_multi_bert = results['Multi_BERT']['model'].predict(X_multi_all_sc)
    
    # Boundaries (Using Multi-Class numbers for the chart labels)
    min_w = int(df_test_multi.loc[multi_idx, 'Word_Count'].min())
    max_w = int(df_test_multi.loc[multi_idx, 'Word_Count'].max())
    boundaries = f"{min_w}-{max_w}"
    
    # Feature importance for this bin
    quick_model = xgb.XGBClassifier(tree_method='hist', device=DEVICE, n_estimators=100, max_depth=6, random_state=SEED, n_jobs=MAX_WORKERS)
    quick_model.fit(X_multi_all_sc, df_test_multi.loc[multi_idx, 'is_AI']) 
    imp_df = pd.DataFrame({'Feature': X_multi_all.columns, 'Importance': quick_model.feature_importances_})
    top_5_feats = ", ".join(imp_df.sort_values('Importance', ascending=False).head(5)['Feature'].tolist())
    
    t5_data.append({
        'Length Bin': bin_label, 
        'Boundaries (Words)': boundaries,
        'N samples': len(multi_idx),
        'Bin Acc (Features-only)': accuracy_score(y_bin_true, p_bin_nobert),
        'Bin F1 (Features-only)': precision_recall_fscore_support(y_bin_true, p_bin_nobert, average='macro', zero_division=0)[2],
        'Bin Acc (Features+BERT)': accuracy_score(y_bin_true, p_bin_bert),
        'Bin F1 (Features+BERT)': precision_recall_fscore_support(y_bin_true, p_bin_bert, average='macro', zero_division=0)[2],
        'Multi Acc (Features-only)': accuracy_score(y_multi_true, p_multi_nobert),
        'Multi F1 (Features-only)': precision_recall_fscore_support(y_multi_true, p_multi_nobert, average='macro', zero_division=0)[2],
        'Multi Acc (Features+BERT)': accuracy_score(y_multi_true, p_multi_bert),
        'Multi F1 (Features+BERT)': precision_recall_fscore_support(y_multi_true, p_multi_bert, average='macro', zero_division=0)[2],
        'Top 5 Features': top_5_feats
    })

t5_df = pd.DataFrame(t5_data)
t5_df.round(4).to_csv('results/tables/T5.csv', index=False)

# --- PLOT F9 ---
custom_labels = [f"{row['Length Bin']}\n({row['Boundaries (Words)']} words)\nN = {row['N samples']}" for _, row in t5_df.iterrows()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), dpi=300)
x = np.arange(3)
width = 0.35

ax1.bar(x - width/2, t5_df['Bin F1 (Features-only)'], width, label='Features Only', color='steelblue')
ax1.bar(x + width/2, t5_df['Bin F1 (Features+BERT)'], width, label='Features + BERT', color='skyblue')
ax1.set_xticks(x)
ax1.set_xticklabels(custom_labels) 
ax1.set_title('Binary Detection F1 by Length', fontweight='bold')
ax1.set_ylabel('F1-Score')
ax1.set_ylim([0, 1.1])
ax1.legend()
ax1.grid(axis='y', linestyle='--', alpha=0.5)

ax2.bar(x - width/2, t5_df['Multi F1 (Features-only)'], width, label='Features Only', color='coral')
ax2.bar(x + width/2, t5_df['Multi F1 (Features+BERT)'], width, label='Features + BERT', color='lightsalmon')
ax2.set_xticks(x)
ax2.set_xticklabels(custom_labels) 
ax2.set_title('Multi-class Attribution F1 by Length', fontweight='bold')
ax2.set_ylabel('F1-Score')
ax2.set_ylim([0, 1.1])
ax2.legend()
ax2.grid(axis='y', linestyle='--', alpha=0.5)

plt.suptitle('F9: Length Sensitivity Analysis', fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('results/figures/F9.png', bbox_inches='tight')
plt.close()
print("✓ Task 1.3 Complete. Table T5 and Figure F9 saved.")


In [ ]:
# ==========================================
# CELL 6: TASK 1.4 - CROSS DOMAIN
# ==========================================
print("Running Cross-Domain Evaluation (Per-Class Metrics)...")
cd_files = {
    'MGT Essay (GPT4All vs Human)': 'data/processed/processed_cross_domain_essay.csv',
    'MGT WP (GPT4All vs Human)': 'data/processed/processed_cross_domain_wp.csv',
    'Reuters/Claude (unseen model)': 'data/processed/processed_unseen_reuters.csv'
}

t6_data = []
train_features_nobert = results['Bin_NoBERT']['features']
train_features_all = results['Bin_BERT']['features']

for name, path in cd_files.items():
    df_cd = pd.read_csv(path)
    y_cd = df_cd['is_AI']
    
    X_cd_nobert = df_cd.reindex(columns=train_features_nobert, fill_value=0)
    X_cd_all = df_cd.reindex(columns=train_features_all, fill_value=0)
    
    # Scale
    X_all_sc = pd.DataFrame(results['Bin_BERT']['scaler'].transform(X_cd_all), columns=X_cd_all.columns).astype('float32')
    
    # Predict (BERT Model only for detailed stats to keep table clean)
    p_bert = results['Bin_BERT']['model'].predict(X_all_sc)
    
    # Extract Per-Class Metrics (0=Human, 1=AI)
    prec, rec, f1, _ = precision_recall_fscore_support(y_cd, p_bert, labels=[0, 1], zero_division=0)
    
    t6_data.append({
        'Benchmark': name,
        'Accuracy': accuracy_score(y_cd, p_bert),
        'Human Precision': prec[0], 'Human Recall': rec[0], 'Human F1': f1[0],
        'AI Precision': prec[1], 'AI Recall': rec[1], 'AI F1': f1[1]
    })
    
    # F10 & F11 
    if 'Essay' in name:
        plot_cm(y_cd, p_bert, ['Human', 'AI'], 'F10.png', 'F10: Essay Benchmark CM')
    elif 'WP' in name:
        plot_cm(y_cd, p_bert, ['Human', 'AI'], 'F11a.png', 'F11a: WP Benchmark CM')
    elif 'Reuters' in name:
        plot_cm(y_cd, p_bert, ['Human', 'AI'], 'F11b.png', 'F11b: Reuters Unseen CM')

pd.DataFrame(t6_data).round(4).to_csv('results/tables/T6.csv', index=False)
print("✓ Task 1.4 Complete. Table T6 (Per-Class) and Figures F10, F11 saved.")



with open('results/models/Bin_NoBERT_metadata.json', 'r') as f:
    meta = json.load(f)
train_features_nobert = meta['feature_list']

scaler = joblib.load('results/models/Bin_NoBERT_scaler.joblib')
model = joblib.load('results/models/Bin_NoBERT_model.joblib')

t6_nobert_data = []

for name, path in cd_files.items():
    df_cd = pd.read_csv(path)
    y_cd = df_cd['is_AI']
    
    X_cd_nobert = df_cd.reindex(columns=train_features_nobert, fill_value=0)
    
    X_nobert_sc = pd.DataFrame(scaler.transform(X_cd_nobert), columns=X_cd_nobert.columns).astype('float32')
    
    p_nobert = model.predict(X_nobert_sc)
    
    prec, rec, f1, _ = precision_recall_fscore_support(y_cd, p_nobert, labels=[0, 1], zero_division=0)
    
    t6_nobert_data.append({
        'Benchmark': name,
        'Accuracy': f"{accuracy_score(y_cd, p_nobert):.4f}",
        'Human Precision': f"{prec[0]:.4f}", 'Human Recall': f"{rec[0]:.4f}", 'Human F1': f"{f1[0]:.4f}",
        'AI Precision': f"{prec[1]:.4f}", 'AI Recall': f"{rec[1]:.4f}", 'AI F1': f"{f1[1]:.4f}"
    })

df_nobert = pd.DataFrame(t6_nobert_data)
print("FEATURES ONLY RESULTS:")
print("="*80)
df_nobert.to_csv("results/tables/T13_FEATURES_ONLY_RESULTS")


In [ ]:
# ==========================================
# CELL 7: TASK 1.5 - ADVERSARIAL ATTACKS
# ==========================================
print("Running Adversarial Attacks...")
adv_files = {
    'Paraphrasing (Gemma-27b)': 'data/processed/processed_paraphrased_articles.csv',
    'Translation (NLLB)': 'data/processed/processed_translation.csv'
}

t7_data = [{
    'Attack': 'None (baseline)',
    'Features Acc': results['Bin_NoBERT']['acc'], 'Feat+BERT Acc': results['Bin_BERT']['acc'],
    'Features F1': results['Bin_NoBERT']['f1'].mean(), 'Feat+BERT F1': results['Bin_BERT']['f1'].mean()
}]

for name, path in adv_files.items():
    df_adv = pd.read_csv(path)
    
    # BINARY EVALUATION
    y_adv = df_adv['is_AI']
    
    X_adv_nobert = df_adv.reindex(columns=train_features_nobert, fill_value=0)
    X_adv_all = df_adv.reindex(columns=train_features_all, fill_value=0)
    
    X_nobert_sc = pd.DataFrame(results['Bin_NoBERT']['scaler'].transform(X_adv_nobert), columns=X_adv_nobert.columns).astype('float32')
    X_all_sc = pd.DataFrame(results['Bin_BERT']['scaler'].transform(X_adv_all), columns=X_adv_all.columns).astype('float32')
    
    p_nobert = results['Bin_NoBERT']['model'].predict(X_nobert_sc)
    p_bert = results['Bin_BERT']['model'].predict(X_all_sc)
    
    t7_data.append({
        'Attack': name,
        'Features Acc': accuracy_score(y_adv, p_nobert), 'Feat+BERT Acc': accuracy_score(y_adv, p_bert),
        'Features F1': precision_recall_fscore_support(y_adv, p_nobert, average='macro', zero_division=0)[2],
        'Feat+BERT F1': precision_recall_fscore_support(y_adv, p_bert, average='macro', zero_division=0)[2]
    })
    
    # MULTI-CLASS EVALUATION & CONFUSION MATRICES
    if 'Writer' in df_adv.columns:
        df_adv_multi = df_adv.copy()
        df_adv_multi['Writer'] = df_adv_multi['Writer'].str.replace(r'(_translation|-translation|_paraphrased|-paraphrased)', '', regex=True)
        
        valid_idx = df_adv_multi['Writer'].isin(multi_classes)
        df_adv_multi = df_adv_multi[valid_idx]
        
        if not df_adv_multi.empty:
            y_multi_adv = le_multi.transform(df_adv_multi['Writer'])
            X_adv_multi = df_adv_multi.reindex(columns=train_features_all, fill_value=0)
            
            X_multi_sc = pd.DataFrame(results['Multi_BERT']['scaler'].transform(X_adv_multi), columns=X_adv_multi.columns).astype('float32')
            p_multi_bert = results['Multi_BERT']['model'].predict(X_multi_sc)
            
            # Generate CM for BOTH Paraphrasing and Translation
            clean_name = "Paraphrasing" if "Paraphrasing" in name else "Translation"
            plot_cm(y_multi_adv, p_multi_bert, multi_classes, f'F12_{clean_name}.png', f'F12: {clean_name} Attack (Multi-class)')
            print(f"✓ Confusion Matrix for {clean_name} generated successfully.")

pd.DataFrame(t7_data).round(4).to_csv('results/tables/T7.csv', index=False)
print("✓ Task 1.5 Complete. Table T7 saved.")

In [ ]:
# ==========================================
# CELL 8: TASK 1.6 - FEATURE ABLATION
# ==========================================
print("Running Feature Ablation Study (This will take a few minutes)...")
FEATURE_GROUPS = {
    "Perplexity + UID": ["ppl", "uid"],
    "Burstiness": ["burstiness"],
    "TTR + Stylometry": ["ttr", "complex", "avg_word_len", "avg_sent_len"],
    "LIWC/Empath": ['gain', 'beauty', 'government', 'urban', 'art', 'help', 'optimism', 'strength', 'love', 'traveling'],
    "Semantic Consistency": ["semantic_mean", "semantic_std"],
    "Syntax Depth": ["syntax_depth"],
    "BERT Embeddings": bert_cols
}

t8_data = [{
    'Removed Group': 'None (full model)',
    'Binary Acc': results['Bin_BERT']['acc'], 'Binary F1': results['Bin_BERT']['f1'].mean(),
    'Multi Acc': results['Multi_BERT']['acc'], 'Multi F1': results['Multi_BERT']['f1'].mean()
}]

X_abl = df.drop(columns=['is_AI', 'Writer', 'Text'], errors='ignore')
y_bin_abl = df['is_AI']
y_multi_abl = le_multi.transform(df['Writer'])

for group, cols in FEATURE_GROUPS.items():
    print(f"  - Ablating {group}...")
    X_dropped = X_abl.drop(columns=cols, errors='ignore')
    
    bin_res = optimized_trainer(X_dropped, y_bin_abl, 'binary', True)
    mul_res = optimized_trainer(X_dropped, y_multi_abl, 'multiclass', True)
    
    t8_data.append({
        'Removed Group': group,
        'Binary Acc': bin_res['acc'], 'Binary F1': bin_res['f1'].mean(),
        'Multi Acc': mul_res['acc'], 'Multi F1': mul_res['f1'].mean()
    })
    
    if group == "Burstiness":
        print("    [Check] Running Burstiness ablation with different seeds to verify stability...")
        seeds_to_test = [42, 1234]
        reproducibility_acc = []
        for s in seeds_to_test:
            X_tr, X_te, y_tr, y_te = train_test_split(X_dropped, y_bin_abl, test_size=0.20, random_state=s, stratify=y_bin_abl)
            sc = StandardScaler()
            X_tr_sc = pd.DataFrame(sc.fit_transform(X_tr).astype('float32'), columns=X_dropped.columns)
            X_te_sc = pd.DataFrame(sc.transform(X_te).astype('float32'), columns=X_dropped.columns)
            
            test_model = xgb.XGBClassifier(objective='binary:logistic', tree_method='hist', device=DEVICE, n_estimators=500, learning_rate=0.01, max_depth=10, random_state=s, n_jobs=MAX_WORKERS)
            test_model.fit(X_tr_sc, y_tr, verbose=False)
            p = test_model.predict(X_te_sc)
            reproducibility_acc.append(accuracy_score(y_te, p))
            
        print(f"    [Result] Burstiness Accuracies -> Seed 1 (42): {reproducibility_acc[0]:.4f} | Seed 2 (1234): {reproducibility_acc[1]:.4f}")

t8_df = pd.DataFrame(t8_data)
t8_df.round(4).to_csv('results/tables/T8.csv', index=False)

t8_df['Bin Drop'] = t8_df['Binary F1'].iloc[0] - t8_df['Binary F1']
t8_df['Multi Drop'] = t8_df['Multi F1'].iloc[0] - t8_df['Multi F1']
plot_df = t8_df.iloc[1:].sort_values('Bin Drop', ascending=True)

plt.figure(figsize=(10, 8), dpi=300)
y_pos = np.arange(len(plot_df))
height = 0.35
plt.barh(y_pos - height/2, plot_df['Bin Drop'], height, label='Binary F1 Drop')
plt.barh(y_pos + height/2, plot_df['Multi Drop'], height, label='Multi-class F1 Drop')
plt.yticks(y_pos, plot_df['Removed Group'])
plt.xlabel('Decrease in F1-Score')
plt.title('F13: Feature Ablation Impact')
plt.legend()
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.savefig('results/figures/F13.png', bbox_inches='tight')
plt.close()

print("✓ Task 1.6 Complete. Table T8 and Figure F13 saved.")

In [ ]:
# ==========================================
# CELL 9: BINARY STATISTICAL SIGNIFICANCE (5 SEEDS)
# ==========================================
from sklearn.decomposition import PCA

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

print("="*60)
print("STARTING FAST 5-SEED EVALUATION")
print("="*60)

df = pd.read_csv('data/processed/processed_articles.csv')
bert_cols = [c for c in df.columns if c.startswith('bert_')]
X_all = df.drop(columns=['is_AI', 'Writer', 'Text'], errors='ignore')
y_bin = df['is_AI']

DEVICE = 'cuda'
MAX_WORKERS = 4
seeds_list = [42, 123, 999, 2024, 8080]

detailed_data = []
stats_results = []

for s in seeds_list:
    print(f" -> Training Seed {s}...")
    human_idx = y_bin[y_bin == 0].index
    ai_idx = y_bin[y_bin == 1].index
    n_human = len(human_idx)
    
    random_ai_idx = np.random.RandomState(s).choice(ai_idx, size=n_human, replace=False)
    balanced_idx = np.concatenate([human_idx, random_ai_idx])
    
    X_bal = X_all.loc[balanced_idx]
    y_bal = y_bin.loc[balanced_idx]
    
    X_tr, X_te, y_tr, y_te = train_test_split(X_bal, y_bal, test_size=0.20, random_state=s, stratify=y_bal)
    
    sc = StandardScaler()
    X_tr_sc = pd.DataFrame(sc.fit_transform(X_tr).astype('float32'), columns=X_all.columns)
    X_te_sc = pd.DataFrame(sc.transform(X_te).astype('float32'), columns=X_all.columns)

    X_tr_nobert = X_tr_sc.drop(columns=bert_cols)
    X_te_nobert = X_te_sc.drop(columns=bert_cols)

    # Features only model
    m_no = xgb.XGBClassifier(objective='binary:logistic', tree_method='hist', device=DEVICE, n_estimators=2000, early_stopping_rounds=100, learning_rate=0.01, max_depth=10, random_state=s, n_jobs=MAX_WORKERS,subsample=0.8,colsample_bytree=0.8)
    m_no.fit(X_tr_nobert, y_tr, eval_set=[(X_te_nobert, y_te)], verbose=False)
    p_no = m_no.predict(X_te_nobert)
    acc_no = accuracy_score(y_te, p_no)
    f1_no = precision_recall_fscore_support(y_te, p_no, average='macro', zero_division=0)[2]
    
    # Features + BERT model
    m_b = xgb.XGBClassifier(objective='binary:logistic', tree_method='hist', device=DEVICE, n_estimators=2000, early_stopping_rounds=100, learning_rate=0.01, max_depth=10, random_state=s, n_jobs=MAX_WORKERS,subsample=0.8,colsample_bytree=0.8)
    m_b.fit(X_tr_sc, y_tr, eval_set=[(X_te_sc, y_te)], verbose=False)
    p_b = m_b.predict(X_te_sc)
    acc_b = accuracy_score(y_te, p_b)
    f1_b = precision_recall_fscore_support(y_te, p_b, average='macro', zero_division=0)[2]

    stats_results.append({'Seed': s, 'NoBERT_Acc': acc_no, 'NoBERT_F1': f1_no, 'BERT_Acc': acc_b, 'BERT_F1': f1_b})
    
    # save detailed results for this seed
    detailed_data.append({'Experiment': f"Seed {s}", 'Model': 'Features Only', 'Accuracy': f"{acc_no:.4f}", 'F1-Score': f"{f1_no:.4f}"})
    detailed_data.append({'Experiment': f"Seed {s}", 'Model': 'Features + BERT', 'Accuracy': f"{acc_b:.4f}", 'F1-Score': f"{f1_b:.4f}"})

detailed_data.append({'Experiment': '-'*15, 'Model': '-'*15, 'Accuracy': '-'*15, 'F1-Score': '-'*15})

stats_df = pd.DataFrame(stats_results)
mean_df = stats_df.mean()
std_df = stats_df.std()

detailed_data.append({
    'Experiment': 'FINAL SUMMARY', 'Model': 'Features Only',
    'Accuracy': f"{mean_df['NoBERT_Acc']:.4f} ± {std_df['NoBERT_Acc']:.4f}",
    'F1-Score': f"{mean_df['NoBERT_F1']:.4f} ± {std_df['NoBERT_F1']:.4f}"
})
detailed_data.append({
    'Experiment': 'FINAL SUMMARY', 'Model': 'Features + BERT',
    'Accuracy': f"{mean_df['BERT_Acc']:.4f} ± {std_df['BERT_Acc']:.4f}",
    'F1-Score': f"{mean_df['BERT_F1']:.4f} ± {std_df['BERT_F1']:.4f}"
})

pd.DataFrame(detailed_data).to_csv('results/tables/T9_Statistical_Significance_Detailed.csv', index=False)
print("="*60)
print("✓ Detailed Statistical Significance Test saved to T9_Statistical_Significance_Detailed.csv")

In [ ]:
# ==========================================
# CELL 10: PCA VISUALIZATION
# ==========================================

print("\n[2/3] Generating PCA Feature Space Visualizations...")
pca = PCA(n_components=2, random_state=SEED)

X_nobert_pca = pca.fit_transform(results['Bin_NoBERT']['scaler'].transform(X_no_bert))
X_all_pca = pca.fit_transform(results['Bin_BERT']['scaler'].transform(X_all))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), dpi=300)

sns.scatterplot(x=X_nobert_pca[:,0], y=X_nobert_pca[:,1], hue=df['Writer'], palette='tab10', s=15, alpha=0.6, ax=ax1)
ax1.set_title('PCA: Features Only', fontweight='bold')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)

sns.scatterplot(x=X_all_pca[:,0], y=X_all_pca[:,1], hue=df['Writer'], palette='tab10', s=15, alpha=0.6, ax=ax2)
ax2.set_title('PCA: Features + BERT', fontweight='bold')
ax2.get_legend().remove()

plt.suptitle('F14: PCA Dimensionality Reduction', fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('results/figures/F14_PCA.png', bbox_inches='tight')
plt.close()
print("  ✓ PCA Plot saved to F14_PCA.png")

In [ ]:
# ==========================================
# CELL 11: PER-CLASS F1 FOR ALL ATTACKS
# ==========================================

# PER-CLASS F1 FOR ALL ATTACKS (T10)
print("\n[3/3] Calculating Per-Class F1 for Attacks...")
attack_files_multi = {
    'Misspelling (5%)': 'data/processed/processed_misspelled_5.csv',
    'Misspelling (10%)': 'data/processed/processed_misspelled_10.csv',
    'Misspelling (15%)': 'data/processed/processed_misspelled_15.csv',
    'Misspelling (20%)': 'data/processed/processed_misspelled_20.csv',
    'Translation': 'data/processed/processed_translation.csv',
    'Paraphrasing': 'data/processed/processed_paraphrased_articles.csv'
}

per_class_results = []
for atk_name, atk_path in attack_files_multi.items():
    try:
        df_atk = pd.read_csv(atk_path)
        if 'Writer' in df_atk.columns:
            df_atk['Writer'] = df_atk['Writer'].str.replace(r'(_translation|-translation|_paraphrased|-paraphrased)', '', regex=True)
            valid_idx = df_atk['Writer'].isin(multi_classes)
            df_atk = df_atk[valid_idx]

            if not df_atk.empty:
                y_true = le_multi.transform(df_atk['Writer'])
                X_atk = df_atk.reindex(columns=train_features_all, fill_value=0)
                X_atk_sc = pd.DataFrame(results['Multi_BERT']['scaler'].transform(X_atk), columns=X_atk.columns).astype('float32')

                p_pred = results['Multi_BERT']['model'].predict(X_atk_sc)
                f1_scores = precision_recall_fscore_support(y_true, p_pred, labels=range(len(multi_classes)), average=None, zero_division=0)[2]

                row = {'Attack Type': atk_name}
                for i, cls_name in enumerate(multi_classes):
                    row[cls_name] = f1_scores[i]
                per_class_results.append(row)
    except Exception as e:
        print(f"  ⚠️ Skipping {atk_name} due to an error: {e}")

if per_class_results:
    df_t10 = pd.DataFrame(per_class_results).round(4)
    # Arrange logic for neat output
    order_map = {'Misspelling (5%)': 1, 'Misspelling (10%)': 2, 'Misspelling (15%)': 3, 'Misspelling (20%)': 4, 'Paraphrasing': 5, 'Translation': 6}
    df_t10['Order'] = df_t10['Attack Type'].map(lambda x: order_map.get(x, 99))
    df_t10 = df_t10.sort_values('Order').drop(columns=['Order']).reset_index(drop=True)
    df_t10.to_csv('results/tables/T10_PerClass_Attacks.csv', index=False)
    print("  ✓ Per-Class F1 for Attacks saved to T10_PerClass_Attacks.csv")

print("\n" + "="*60)
print("🎉 ALL OVERNIGHT TASKS COMPLETED SUCCESSFULLY! 🎉")
print("="*60)

In [ ]:
# ==========================================
# CELL 12: MULTI-CLASS STATISTICAL SIGNIFICANCE (5 SEEDS)
# ==========================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')

print('='*60)
print('STARTING MULTI-CLASS 5-SEED EVALUATION')
print('='*60)

# Load data afresh to maintain independence
df = pd.read_csv('data/processed/processed_articles.csv')
bert_cols = [c for c in df.columns if c.startswith('bert_')]
X_all = df.drop(columns=['is_AI', 'Writer', 'Text'], errors='ignore')
y_multi = le_multi.transform(df['Writer'])

DEVICE_CELL = globals().get('DEVICE', 'cuda')
MAX_WORKERS_CELL = globals().get('MAX_WORKERS', 4)
seeds_list = [42, 123, 999, 2024, 8080]

detailed_data_multi = []
stats_results_multi = []

per_class_f1_nobert = {cls: [] for cls in multi_classes}
per_class_f1_bert = {cls: [] for cls in multi_classes}

for s in seeds_list:
    print(f' -> Training Multi-Class Seed {s}...')
    
    # Train-test split inside loop to evaluate statistical significance across different partitions
    X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(
        X_all, y_multi, test_size=0.20, random_state=s, stratify=y_multi
    )
    
    # Standardize features
    sc_m = StandardScaler()
    X_tr_sc_m = pd.DataFrame(sc_m.fit_transform(X_tr_m).astype('float32'), columns=X_all.columns)
    X_te_sc_m = pd.DataFrame(sc_m.transform(X_te_m).astype('float32'), columns=X_all.columns)
    
    # Create No-BERT feature sets
    X_tr_nobert_m = X_tr_sc_m.drop(columns=bert_cols)
    X_te_nobert_m = X_te_sc_m.drop(columns=bert_cols)
    
    # Model 1: Features Only (No BERT)
    m_no_multi = xgb.XGBClassifier(
        objective='multi:softprob',
        tree_method='hist',
        device=DEVICE_CELL,
        n_estimators=2000,
        early_stopping_rounds=100,
        learning_rate=0.01,
        max_depth=10,
        random_state=s,
        n_jobs=MAX_WORKERS_CELL,
        subsample=0.8,
        colsample_bytree=0.8
    )
    m_no_multi.fit(X_tr_nobert_m, y_tr_m, eval_set=[(X_te_nobert_m, y_te_m)], verbose=False)
    p_no_multi = m_no_multi.predict(X_te_nobert_m)
    
    acc_no_multi = accuracy_score(y_te_m, p_no_multi)
    f1_no_multi = precision_recall_fscore_support(y_te_m, p_no_multi, average='macro', zero_division=0)[2]
    f1_no_multi_per_class = precision_recall_fscore_support(y_te_m, p_no_multi, average=None, zero_division=0)[2]
    
    # Model 2: Features + BERT
    m_b_multi = xgb.XGBClassifier(
        objective='multi:softprob',
        tree_method='hist',
        device=DEVICE_CELL,
        n_estimators=2000,
        early_stopping_rounds=100,
        learning_rate=0.01,
        max_depth=10,
        random_state=s,
        n_jobs=MAX_WORKERS_CELL,
        subsample=0.8,
        colsample_bytree=0.8
    )
    m_b_multi.fit(X_tr_sc_m, y_tr_m, eval_set=[(X_te_sc_m, y_te_m)], verbose=False)
    p_b_multi = m_b_multi.predict(X_te_sc_m)
    
    acc_b_multi = accuracy_score(y_te_m, p_b_multi)
    f1_b_multi = precision_recall_fscore_support(y_te_m, p_b_multi, average='macro', zero_division=0)[2]
    f1_b_multi_per_class = precision_recall_fscore_support(y_te_m, p_b_multi, average=None, zero_division=0)[2]
    
    stats_results_multi.append({
        'Seed': s, 'NoBERT_Acc': acc_no_multi, 'NoBERT_F1': f1_no_multi,
        'BERT_Acc': acc_b_multi, 'BERT_F1': f1_b_multi
    })
    
    for i, cls in enumerate(multi_classes):
        per_class_f1_nobert[cls].append(f1_no_multi_per_class[i])
        per_class_f1_bert[cls].append(f1_b_multi_per_class[i])
    
    detailed_data_multi.append({'Experiment': f'Seed {s}', 'Model': 'Features Only', 'Accuracy': f'{acc_no_multi:.4f}', 'F1-Score': f'{f1_no_multi:.4f}'})
    detailed_data_multi.append({'Experiment': f'Seed {s}', 'Model': 'Features + BERT', 'Accuracy': f'{acc_b_multi:.4f}', 'F1-Score': f'{f1_b_multi:.4f}'})

# Calculate Final Statistics
detailed_data_multi.append({'Experiment': '-'*15, 'Model': '-'*15, 'Accuracy': '-'*15, 'F1-Score': '-'*15})

stats_df_multi = pd.DataFrame(stats_results_multi)
mean_df_multi = stats_df_multi.mean()
std_df_multi = stats_df_multi.std()

detailed_data_multi.append({
    'Experiment': 'FINAL SUMMARY', 'Model': 'Features Only',
    'Accuracy': f"{mean_df_multi['NoBERT_Acc']:.4f} ± {std_df_multi['NoBERT_Acc']:.4f}",
    'F1-Score': f"{mean_df_multi['NoBERT_F1']:.4f} ± {std_df_multi['NoBERT_F1']:.4f}"
})
detailed_data_multi.append({
    'Experiment': 'FINAL SUMMARY', 'Model': 'Features + BERT',
    'Accuracy': f"{mean_df_multi['BERT_Acc']:.4f} ± {std_df_multi['BERT_Acc']:.4f}",
    'F1-Score': f"{mean_df_multi['BERT_F1']:.4f} ± {std_df_multi['BERT_F1']:.4f}"
})

output_path = 'results/tables/T11_Statistical_Significance_Multi_Detailed.csv'
pd.DataFrame(detailed_data_multi).to_csv(output_path, index=False)

# Build and Save Per-Class Summary
per_class_summary = []
for cls in multi_classes:
    nb_mean, nb_std = np.mean(per_class_f1_nobert[cls]), np.std(per_class_f1_nobert[cls], ddof=1)
    b_mean, b_std = np.mean(per_class_f1_bert[cls]), np.std(per_class_f1_bert[cls], ddof=1)
    per_class_summary.append({
        'Class': cls,
        'Features Only (Mean ± SD)': f'{nb_mean:.4f} ± {nb_std:.4f}',
        'Features + BERT (Mean ± SD)': f'{b_mean:.4f} ± {b_std:.4f}'
    })

per_class_df = pd.DataFrame(per_class_summary)
per_class_path = 'results/tables/T11_Per_Class_5Seeds_Summary.csv'
per_class_df.to_csv(per_class_path, index=False)

print('='*60)
print(f'✓ Overall Multi-Class Statistical Significance Test saved to: \n  {output_path}')
print(f'✓ Per-Class Mean ± SD saved to: \n  {per_class_path}')
print('='*60)
print(per_class_df.to_string(index=False))
